In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
systems = ['language', 'MD', 'DMN']
loc_tasks = [
    [f'langloc_DiffTasks_{i}' for i in range(1, 7)],
    ['spatialFIN'],
    ['spatialFIN']
]
tasks = [f'langloc_DiffTasks_{i}' for i in range(1, 7)]

data = []
for system_i, system in enumerate(systems):
    for loc_task in loc_tasks[system_i]:
        for j1, task1 in enumerate(tasks):
            for j2, task2 in enumerate(tasks):
                for localizer_run in ['1', '2', 'All']:
                    pth = f'./data/{system}_{loc_task}_{task1}_{task2}_{localizer_run}.csv'
                    if not os.path.exists(pth):
                        continue
                    dat = pd.read_csv(pth)
                    dat['System'] = system
                    dat['Localizer'] = np.nan if system in ['MD', 'DMN'] else loc_task.replace('langloc_DiffTasks_', 'V')
                    dat['LocalizerRun'] = localizer_run
                    dat['Task1'] = task1.replace('langloc_DiffTasks_', 'V')
                    dat['Task2'] = task2.replace('langloc_DiffTasks_', 'V')
                    dat[['Effect1', 'Effect2']] = dat['Effects'].str.split('/', expand=True)
                    dat = dat.drop(columns='Effects')
                    data.append(dat)

data = pd.concat(data)
data.loc[data['System'] == 'language', 'ROI'] = data['ROI'][data['System'] == 'language'].replace({
    1: 'LH_IFG_orb', 2: 'LH_IFG', 3: 'LH_MFG', 4: 'LH_AntTemp', 5: 'LH_PostTemp', 6: 'LH_AngG',
    7: 'RH_IFG_orb', 8: 'RH_IFG', 9: 'RH_MFG', 10: 'RH_AntTemp', 11: 'RH_PostTemp', 12: 'RH_AngG'})
data.loc[data['System'] == 'MD', 'ROI'] = data['ROI'][data['System'] == 'MD'].replace({
    1: 'LH_postParietal', 2: 'LH_midParietal', 3: 'LH_antParietal', 4: 'LH_supFrontal',
    5: 'LH_Precentral_A_precG', 6: 'LH_Precentral_B_IFGop', 7: 'LH_midFrontal',
    8: 'LH_midFrontalOrb', 9: 'LH_insula', 10: 'LH_medialFrontal',
    11: 'RH_postParietal', 12: 'RH_midParietal', 13: 'RH_antParietal', 14: 'RH_supFrontal', 
    15: 'RH_Precentral_A_precG', 16: 'RH_Precentral_B_IFGop', 17: 'RH_midFrontal',
    18: 'RH_midFrontalOrb', 19: 'RH_insula', 20: 'RH_medialFrontal'
})
data.loc[data['System'] == 'DMN', 'ROI'] = data['ROI'][data['System'] == 'DMN'].replace({
    1: 'LH_FrontalMed', 2: 'LH_PostCing', 3: 'LH_TPJ', 4: 'LH_MidCing', 5: 'LH_STGorInsula', 6: 'LH_AntTemp',
    7: 'RH_FrontalMed', 8: 'RH_PostCing', 9: 'RH_TPJ', 10: 'RH_MidCing', 11: 'RH_STGorInsula', 12: 'RH_AntTemp'
})
data[['Run1', 'Effect1']] = data['Effect1'].str.split('_', expand=True)
data[['Run2', 'Effect2']] = data['Effect2'].str.split('_', expand=True)
data['Run1'] = data['Run1'].replace({'ODD': '1', 'EVEN': '2'})
data['Run2'] = data['Run2'].replace({'ODD': '1', 'EVEN': '2'})
data = data.rename(columns={'FisherTransformedCorrelationCoefficients': 'FisherZ'})
data['Hemisphere'] = data['ROI'].str.split('_').str[0]
data = data[['Subject', 'System', 'Hemisphere', 'ROI', 
      'Localizer', 'LocalizerRun', 
      'Task1', 'Run1', 'Effect1', 
      'Task2', 'Run2', 'Effect2', 'FisherZ']]

/var/folders/k0/md38zs8x1395ywbtngppz3k00000gp/T/ipykernel_82523/353031483.py:29: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['LH_IFG_orb' 'LH_IFG_orb' 'LH_IFG_orb' ... 'RH_AngG' 'RH_AngG' 'RH_AngG']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[data['System'] == 'language', 'ROI'] = data['ROI'][data['System'] == 'language'].replace({


In [3]:
data['Type1'] = data.apply(
    lambda row: 'TwoSentences' if row['Task1'] in ['V4','V5','V6'] and row['Effect1'] == 'S' else 'OneSentence',
    axis=1
)
data['Type2'] = data.apply(
    lambda row: 'TwoSentences' if row['Task2'] in ['V4','V5','V6'] and row['Effect2'] == 'S' else 'OneSentence',
    axis=1
)
data['Version1'] = data['Task1']
data['Task1'] = data.apply(
        lambda row: 'ButtonPress' if row['Version1'] == 'V1' or (row['Version1'] == 'V6' and row['Effect1'] == 'S')
        else 'MemoryAll' if row['Version1'] == 'V2' or (row['Version1'] in ['V4','V5','V6'] and row['Effect1'] == 'N')
        else 'MemoryLast' if row['Version1'] == 'V3'
        else 'ComprehensionQ' if row['Version1'] == 'V4'
        else 'SentimentQ',
        axis=1
    )
data['Version2'] = data['Task2']
data['Task2'] = data.apply(
        lambda row: 'ButtonPress' if row['Version2'] == 'V1' or (row['Version2'] == 'V6' and row['Effect2'] == 'S')
        else 'MemoryAll' if row['Version2'] == 'V2' or (row['Version2'] in ['V4','V5','V6'] and row['Effect2'] == 'N')
        else 'MemoryLast' if row['Version2'] == 'V3'
        else 'ComprehensionQ' if row['Version2'] == 'V4'
        else 'SentimentQ',
        axis=1
    )

In [5]:
data.to_csv('/Users/rgao76/Documents/DiffTasks/spatial_correlation_2025/Data/all_data.csv', index=False)